# $\Omega$ and $\beta$ Usage

This notebook shows the minimal workflow for computing $\beta$ from local omega analysis on one smoothed disclination line.

## JupyterLab and PyVistaQt

**If you are not running this workflow in JupyterLab, you usually do not need this setup section.**

This setup section exists ONLY to handle compatibility between JupyterLab and `pyvistaqt`. `pyvistaqt` opens a native Qt window instead of rendering inline inside the notebook. In JupyterLab, the Qt event loop must be enabled before creating that window:

```python
%gui qt
```

The `QT_API` line below chooses a Qt binding before importing `pyvistaqt` or `nematics3d`:

```python
os.environ["QT_API"] = "pyqt5"
```

If you see an incompatible Qt binding error, restart the kernel and run the notebook again from the top. A Qt binding cannot be switched cleanly after it has already been imported in the same kernel.

This also requires a local desktop session with a working Qt backend. It will not work in a headless browser-only server unless an appropriate display is available.

In [1]:
import os
os.environ["QT_API"] = "pyqt5"
%gui qt

## Imports

In [2]:
from pathlib import Path
import sys

import numpy as np


REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src" / "nematics3d").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Could not locate the repository root.")
    REPO_ROOT = REPO_ROOT.parent

SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import nematics3d

## Load Q Data

This notebook starts from a saved Q-tensor field as the example file `Q_1630.npy`, from Vincent's simulation.

After loading the file, the variable `Q_data` should be a 3D lattice of Q tensors. In this example, the first index of the saved array selects one frame, and the crop selects a smaller spatial region around one defect line.

`nematics3d.QFieldObject` is the main entry point for this repository. When it is created from `Q_data`, it detects disclination points and classifies them into line objects. For this example, the crop is chosen so that exactly one line is present.

In [3]:
DATA_PATH = REPO_ROOT / "tests" / "disclination" / "beta" / "Q_1630.npy"
Q_data = np.load(DATA_PATH)[0]
Q_data = Q_data[168:185, 5:32, 10:35]

Q = nematics3d.QFieldObject(
    Q=Q_data,
    name="WT",
)

[PROGRESS]
    <QFieldObject.__init__> 
    Start to initialize Q tensor `WT`.
[PROGRESS]
    <QFieldObject.__init__> 
    Start defect analysis as detecting defects and classifying them into distinct lines for Q tensor `WT` 
    This operation might take a while.
    You can disable this automatic operation by setting is_detect_defects=False and is_classify_lines=False when initializing the Q tensor.
[INFO]
        <QFieldObject[name='WT'].act_defect_detect> 
        70 defects are found.
[INFO]
        <QFieldObject[name='WT'].act_lines_classify> 
        1 lines are found.
[PROGRESS]
    <QFieldObject.__init__> 
    Defect analysis is finished, with 0.01 s


## Smooth the Single Line

Computing $\beta$ requires the local tangent direction of the disclination line at the chosen position. Raw detected defect points are discrete and may be noisy, so the line must be smoothed before `act_calc_omega(...)` can evaluate a stable tangent.

The smoothing routine has several parameters, but the most important one for basic use is `window_length`. This is NOT a physical arc length. It is the number of discrete line samples included in the local smoothing window along the ordered defect-line points. For example, `window_length=28` means the local fit/filter uses a neighborhood of 28 sampled points along the line. A larger window suppresses more small-scale noise, but it can also wash out real curvature.

A later Q&A section will discuss how to visually judge whether the selected smoothing window is appropriate for your data.

In [4]:
if len(Q.lines) != 1:
    raise RuntimeError(f"Expected exactly one line, got {len(Q.lines)}.")

Q.act_lines_smooth(window_length=28)
smooth = Q.lines[0].smooths[-1]
smooth

[INFO]
    <QFieldObject[name='WT'].act_lines_smooth> 
    No input value provided for minimum smoothed line length. 
    Using the default value self.default_miminum_line_length_smooth=61.
[INFO]
    <QFieldObject[name='WT'].act_lines_smooth> 
    There are 1 disclination lines in total, with 1 lines are smoothed.
    The smoothing window length is: 28


DisclinationLineSmooth('disclination line 0 smooth_version 0')

In this cell, `Q.act_lines_smooth(...)` is the method that creates smoothed versions of the disclination lines stored in the `Q` object. It does not smooth only `Q.lines[0]`; it goes through all classified lines in `Q.lines` and smooths the ones that satisfy the length and option checks.

`Q.lines` is the collection of all disclination-line objects detected in this `QFieldObject`. Each line object stores its own smoothed versions in `line.smooths`. This is useful because you can create multiple smoothed versions of the same raw line using different smoothing parameters.

In this example, there is only one line, so we use `Q.lines[0]`. After smoothing, `Q.lines[0].smooths[-1]` selects the most recently created smoothed version. This smoothed line object is the object used below for tangent-based omega and $\beta$ calculations.

## Compute $\beta$ at One Position

`u_percent` is the internal spline parameter of the smoothed line, expressed as a percentage from 0 to 100. The smoothed line is built from the ordered defect-line samples (the defect points detected by winding number); during smoothing, those samples are assigned a normalized parameter `u` that runs from the beginning of the ordered line to the end. `u_percent` is simply `100 * u`. It tells the code where to evaluate the smoothed spline in this normalized parameter domain.

This means `u_percent=0` is the start of the current smoothed-line parameterization, `u_percent=50` is halfway through that parameter domain, and `u_percent=100` is the end. It is not a physical arc length and it is not measured in simulation length units. The local position and tangent used for $\beta$ are evaluated from this smoothed-line spline parameter.

In [5]:
u_percent = 5
omega_result = smooth.act_calc_omega(u_percent)
beta = omega_result["beta"]
beta

83.48768772428835

For the selected `u_percent`, the computed $\beta$ value suggests a wedge-like defect. Let's visually check whether the local director pattern agrees with that interpretation.

The next cell draws the detected disclination line and then plots the director field around the same `u_percent` position.

In [6]:
Q.act_visualize_disclination_lines(
    is_new=True,  # Create a new figure.
    min_line_length=0,  # Keep even short detected lines so this small local example is not filtered out.
    title="WT disclination lines",  # The title (also the name) of the figure
)

Q.act_visualize_n_near_defect(u_percent=u_percent)  # Draw the director field around the selected point on the defect line.

Q.figs.active_fig.act_commit(  # Preset viewing angle chosen by the author for a clear first look; feel free to rotate freely in the interactive window.
  azimuth=73,
  elevation=49,
  roll=-11,
  distance=20,
  focal_point=[ 6.31487462, 15.66747125,  6.04773563],
)

If you run the previous cell multiple times, you may see a warning (and multiple figures). This is usually harmless. Each run creates new Python objects and a new figure while reusing the same requested names, so those names collide with the older objects and figures that are still registered. The system automatically adjusts the new names and prints a warning to let you know.

If everything runs normally and you have not changed the earlier cells in this notebook, the figure should look like this:

![Expected visualization result](example1.png)

**Congratulations! Thank you for making it this far. We have now shown the most basic workflow for measuring and visualizing $\beta$.**

## Optional: Color the Defect Line by a $\beta$ Line Function

One helpful feature is to create a line-function interpolator. It samples a quantity along the smoothed defect line, builds an interpolator from those samples, and attaches the resulting `SmoothedLineFunc` to the smoothed-line object `smooth` through `smooth.linefuncs`.

Here we use that mechanism for $\beta$: the function evaluates $\beta$ at selected `u_percent` positions, and the resulting `beta_func` can later be reused as a scalar field along the same smoothed line.

The code below reassigns `smooth = Q.lines[0].smooths[-1]` on purpose. If you have rerun earlier cells, the line may now contain multiple smoothed versions; selecting `smooths[-1]` keeps this step attached to the most recently created smoothed line.

In [7]:
smooth = Q.lines[0].smooths[-1]
beta_func = smooth.act_create_linefunc(
    func=lambda u: smooth.act_calc_omega(u)["beta"],  # Function to sample: input is u_percent, output is beta.
    u_samples=np.arange(0, 100, 5),  # Sample positions in u_percent.
    name="beta",  # Registry name used to attach this line function under smooth.linefuncs.
)

smooth.linefuncs[-1]

SmoothedLineFunc('beta'), num_samples=20, mode='wrap'

Now that we have a $\beta$ interpolator, we can use it to color the defect line that is already plotted. This turns $\beta$ into the scalar value used by the tube visualization.

In [8]:
smooth.visual_tube.act_commit(
    paint_by="scalars",  # Use scalar coloring instead of a single fixed tube color.
    resolver_source="u_percent",  # Pass each tube point's normalized line position to beta_func.
    scalars=beta_func,  # The line-function interpolator that returns beta along the smoothed line.
    scalar_bar_title="beta",  # Title shown on the scalar color bar.
)


Q.figs.active_fig.act_commit(  # Preset viewing angle chosen by the author for a clear first look; feel free to rotate freely in the interactive window.
  azimuth=71,
  elevation=20,
  roll=2.1,
  distance=43,
  focal_point=[ 3.86305393, 12.81694508, 10.51823688],
)

If everything runs normally and you have not changed the visualization parameters above, the figure should look like this:

![Expected $\beta$-colored visualization result](example_2.png)

In [9]:
smooth.visual_tube.calc_scalars

array([73.22743 , 73.71334 , 74.19924 , 74.68515 , 73.94154 , 71.968414,
       69.995285, 68.02216 , 64.911804, 61.801456, 58.6911  , 55.54705 ,
       52.369305, 49.191555, 46.013805, 43.892662, 41.771515, 39.65037 ,
       38.5512  , 38.474   , 38.3968  , 38.3196  , 40.299957, 42.28031 ,
       44.260666, 46.961277, 50.382122, 53.80298 , 57.223824, 60.49234 ,
       63.760864, 67.02938 , 69.66617 , 71.67122 , 73.67628 , 75.68133 ,
       75.93278 , 76.18423 , 76.43567 , 75.2432  , 72.60682 , 69.970436,
       67.33405 , 62.51465 , 57.69523 , 52.87582 , 47.92325 , 42.83752 ,
       37.751823, 32.666096, 29.313377, 25.960678, 22.60796 , 20.714756,
       20.281063, 19.847374, 19.413683, 21.88272 , 24.351746, 26.820784,
       30.429907, 35.179115, 39.9283  , 44.677505, 49.716072, 54.754612,
       59.79318 , 63.871746, 66.99031 , 70.108864, 73.22743 ],
      dtype=float32)

In [10]:
smooth.visual_tube.raw_coords

array([[ 7.794154  , 20.79557533, 10.32177728],
       [ 7.75503646, 21.03732542, 11.01112347],
       [ 7.74310963, 21.24094673, 11.69608207],
       [ 7.75960944, 21.38165863, 12.3971079 ],
       [ 7.75342974, 21.42695588, 13.13465579],
       [ 7.72457051, 21.39729329, 13.86849586],
       [ 7.72413793, 21.34501298, 14.58008899],
       [ 7.75089606, 21.24193548, 15.2815474 ],
       [ 7.80360895, 21.08342603, 15.96391052],
       [ 7.88104066, 20.86670374, 16.61883574],
       [ 7.96026449, 20.56915091, 17.23859844],
       [ 8.0483871 , 20.17834631, 17.83778272],
       [ 8.17358794, 19.73482882, 18.40155729],
       [ 8.33339513, 19.27141268, 18.89463602],
       [ 8.49468545, 18.73977259, 19.33469287],
       [ 8.64540848, 18.14454332, 19.7099864 ],
       [ 8.80354715, 17.52162897, 20.01001112],
       [ 8.96539365, 16.8864788 , 20.22549747],
       [ 9.12724014, 16.22512668, 20.37906316],
       [ 9.26368805, 15.52422445, 20.47348906],
       [ 9.37937214, 14.80107527, 20.513

In [11]:
print(
    beta_func(0),
    beta_func(100)
)

73.22742995964987 73.22742995964987


In [12]:
len(smooth.visual_tube.raw_coords)

71

In [13]:
smooth.opts

OptsSmooth: the options of DisclinationLineSmooth('disclination line 0 smooth_version 0')
  tag             = 'smooth options'
  window_ratio    = 2.4
  window_length   = 29
  order           = 3
  num_out_ratio   = 1
  mode            = 'wrap'
  min_line_length = 61